<a href="https://colab.research.google.com/github/Johnny-DF26/Aplicando_nlp_para_analise_sentimentos/blob/main/NLP_aplicando_processamento_de_linguagem_natural_para_an%C3%A1lise_de_sentimentos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **<font color='red'> NLP: Processamento de linguagem natural para análise de sentimentos**

## **Introdução: O Desafio da Experiência do Cliente**

No cenário atual de e-commerce e serviços digitais, o volume de feedback dos usuários é massivo. Analisar manualmente milhares de avaliações para identificar problemas ou elogios é inviável e caro.

### **O Problema**
Empresas precisam de uma forma escalável e precisa para monitorar a satisfação dos seus clientes. Identificar rapidamente avaliações negativas pode permitir ações de suporte imediatas, enquanto entender os pontos positivos ajuda a reforçar estratégias de marketing.

### **O Objetivo do Projeto**
Este projeto visa desenvolver um sistema de **Análise de Sentimentos** utilizando técnicas avançadas de **Processamento de Linguagem Natural (NLP)**.

Neste notebook, passaremos por todo o pipeline de Ciência de Dados:
1.  **Limpeza e Normalização:** Tratamento de ruídos, acentuação e simplificação de palavras (Stemming).
2.  **Vetorização:** Transformação de texto em dados numéricos usando técnicas clássicas e modernas.
3.  **Modelagem:** Comparação de performance entre modelos lineares e Redes Neurais de última geração (BERTimbau).
4.  **Avaliação:** Análise criteriosa de métricas para garantir que o modelo seja confiável para decisões de negócio.

# **<font color='blue'> 1. Explorando os dados**

## **1.1. Conhecendo os dados**

In [ ]:
import pandas as pd

In [ ]:
path = 'https://raw.githubusercontent.com/alura-cursos/nlp_analise_sentimento/refs/heads/main/Dados/dataset_avaliacoes.csv'

In [ ]:
df = pd.read_csv(path)

In [ ]:
print('==='*30)
display(df.head())
print('==='*30)

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df['sentimento'].value_counts()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme()
sns.countplot(x='sentimento', data=df, hue='sentimento')
plt.title('Distribuição de Sentimentos')
plt.ylabel('')
plt.xlabel('')
#plt.yticks([])

for i, valor in enumerate(df['sentimento'].value_counts()):
    label = f"{valor:,.0f}".replace(',','.')
    plt.text(i, valor-700, label, ha='center', va='bottom', color='white')

plt.show()

In [ ]:
print('Avaliação Positiva:\n')

df['avaliacao'][0]

In [ ]:
print('Avaliação Negativa:\n')
df['avaliacao'][2]

# **<font color='blue'> 2. Tratamento dos Dados**

## **2.1. Visualizando as palavras mais utilizadas**

In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

In [ ]:
# Juntando todas as palavras
all_words = [text for text in df['avaliacao']]
all_words = ' '.join([text for text in df['avaliacao']])
all_words[0:1000]

In [ ]:
# Rodando a imagem com o rank das palavras
cloud = WordCloud(width=800, height=500, random_state=42, max_font_size=100, collocations=False).generate(all_words)

# Plotagem da figura
plt.figure(figsize=(15,8))
plt.imshow(cloud, interpolation='bilinear')
plt.axis('off')
plt.show()

In [ ]:
df.head()

In [ ]:
avaliacoes_positivas = df.query("sentimento == 'positivo'")['avaliacao']

avaliacoes_positivas.head()

In [ ]:
avaliacoes_negativas = df.query("sentimento == 'negativo'")['avaliacao']

avaliacoes_negativas.head()

In [ ]:
# Juntando todas as palavras
def juntar_palavras(df):
  all_words = [text for text in df]
  all_words = ' '.join([text for text in df])

  return all_words

In [ ]:
palavras_positivas = juntar_palavras(avaliacoes_positivas)
palavras_negativas = juntar_palavras(avaliacoes_negativas)

In [ ]:
palavras_positivas[0:1000]

In [ ]:
palavras_negativas[0:1000]

In [ ]:
# Rodando a imagem com o rank das palavras Positivas
def rank_palavras(dataframe, avaliacao):
    df = dataframe.query(f"sentimento == '{avaliacao}'")['avaliacao']
    palavras_juntas = juntar_palavras(df)

    cloud = WordCloud(width=800, height=500, random_state=42, max_font_size=100, collocations=False).generate(palavras_juntas)

    # Plotagem da figura
    plt.figure(figsize=(15,8))
    plt.imshow(cloud, interpolation='bilinear')
    plt.title(f'Rank de palavras {avaliacao.title()}')
    plt.axis('off')
    plt.show()

In [ ]:
rank_palavras(df, 'positivo')

In [ ]:
rank_palavras(df, 'negativo')


## **2.2. Limpeza dos dados**

In [ ]:
df.head()

In [ ]:
## Copiando o dataframe
df_normalizado = df.copy()

### **2.2.1. Caixa Baixa**

In [ ]:
## Colocando os dados em caixa baixa
df_normalizado['avaliacao_1'] = df_normalizado['avaliacao'].str.lower()

In [ ]:
df_normalizado.head()

### **2.2.2. Acentos**

In [ ]:
!pip install unidecode

In [ ]:
## Tratando as acentuacoes
import unidecode

texto_sem_acentos = [unidecode.unidecode(texto) for texto in df_normalizado['avaliacao_1']]

df_normalizado['avaliacao_2'] = texto_sem_acentos

In [ ]:
df_normalizado.head()

### **2.2.3. Pontuação**

In [ ]:
from nltk import tokenize

In [ ]:
## Retirando pontuações
token_pontuacao = tokenize.WordPunctTokenizer() ## Separa pelas pontuacoes

palavras_pontuacao = []

for pal in df_normalizado['avaliacao_2']:
    texto_tokenizado = token_pontuacao.tokenize(pal)
    nova_frase = [palavra for palavra in texto_tokenizado if palavra.isalpha()]
    palavras_pontuacao.append(' '.join(nova_frase))

df_normalizado['avaliacao_3'] = palavras_pontuacao

In [ ]:
df_normalizado.head()

### **2.2.4. Stop Words**

In [ ]:
import nltk
nltk.download('stopwords')

In [ ]:
## Retirando as StopWords

palavras_irrelevantes = nltk.corpus.stopwords.words('portuguese')

In [ ]:
## Aplicando Stop Words
from nltk import tokenize

palavras_stop_words = []

stop_word = tokenize.WhitespaceTokenizer()

for palavra in df_normalizado['avaliacao_3']:
    palavras_tokenizadas = stop_word.tokenize(palavra)
    palavras_sem_irrelevantes = [palavra for palavra in palavras_tokenizadas if palavra not in palavras_irrelevantes]
    palavras_stop_words.append(' '.join(palavras_sem_irrelevantes))

df_normalizado['avaliacao_4'] = palavras_stop_words

In [ ]:
df_normalizado.head()

In [ ]:
df_normalizado['avaliacao'][3]

In [ ]:
df_normalizado['avaliacao_4'][3]

In [ ]:
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

def classificar_texto(texto, coluna_texto, coluna_classificacao):

    print(f"Treinando e avaliando Modelo {coluna_texto.split('_')[1]} ...")
    # Vetorizando os dados
    vetorizar = CountVectorizer(max_features=50)
    X = vetorizar.fit_transform(texto[coluna_texto])
    y = texto[coluna_classificacao]

    print(f"Shape: {X.shape}")
    # Pré-processando em números
    encoder = LabelEncoder()
    y = encoder.fit_transform(y)

    # Divisão em treino e teste
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=4978)

    # Treinando o modelo
    rl = LogisticRegression(max_iter=1000)
    rl.fit(X_train, y_train)

    # Avaliando o desempenho do modelo
    acuracia = rl.score(X_test, y_test)


    return print(f"Acurácia do modelo com {coluna_texto}: {acuracia:.2f}")

In [ ]:
classificar_texto(df_normalizado, 'avaliacao_1', 'sentimento')

In [ ]:
classificar_texto(df_normalizado, 'avaliacao_2', 'sentimento')

In [ ]:
classificar_texto(df_normalizado, 'avaliacao_3', 'sentimento')

In [ ]:
classificar_texto(df_normalizado, 'avaliacao_4', 'sentimento')

### **2.2.5. Simplificação de Palavras**

In [ ]:
nltk.download('rslp')

In [ ]:
stemmer  = nltk.RSLPStemmer()

stemmer.stem('gostou')

In [ ]:
frase_processada = []

for text in df_normalizado['avaliacao_4']:
    palavras_texto = token_pontuacao.tokenize(text)
    nova_frase = [stemmer.stem(palavra) for palavra in palavras_texto]
    frase_processada.append(' '.join(nova_frase))


In [ ]:
df_normalizado['avaliacao_5'] = frase_processada

In [ ]:
df_normalizado.head()

In [ ]:
classificar_texto(df_normalizado, 'avaliacao_5', 'sentimento')

### **2.2.6. Aplicação de Contexto**

In [ ]:
from nltk import ngrams
from nltk import tokenize

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
texto_exemplo = 'Eu não gostei do produto, pois veio com defeito.'

token_tokenizer = tokenize.WordPunctTokenizer()
frase_token = token_tokenizer.tokenize(texto_exemplo)
pares = list(ngrams(frase_token, 3))

print(pares)

In [ ]:
texto_exemplo = 'Eu não gostei do produto, pois veio com defeito.'
tf_idf = TfidfVectorizer(max_features=50, sublinear_tf=True, ngram_range=(1,3))

tf_vetorizado = tf_idf.fit_transform([texto_exemplo])

### **2.2.7. Salvando DataFrame**

In [ ]:
## Salvando dataframe

df_normalizado.to_csv('/content/drive/MyDrive/CSVs/df_nlp_sentimentos.csv', index=False)

# **<font color='blue'> 3. Aplicando o Algoritmos de NLP**

## **Buscando Base de dados salva**

In [ ]:
import pandas as pd

In [ ]:
df_normalizado = pd.read_csv('/content/drive/MyDrive/CSVs/df_nlp_sentimentos.csv')

## **3.1. Divisão em treino e teste**

In [ ]:
df_normalizado.head()

In [ ]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

df_encoder = encoder.fit_transform(df_normalizado['sentimento'])
df_normalizado['sentimento_encoder'] = df_encoder

In [ ]:
df_normalizado

In [ ]:
df_normalizado.isnull().sum()


In [ ]:
df_normalizado.dropna(inplace=True)

In [ ]:
df_normalizado.isnull().sum()


In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(df_normalizado['avaliacao_5'], df_normalizado['sentimento_encoder'], test_size=0.2, random_state=42)

In [ ]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

## **3.2. Bag of Words**

### 📝 Técnica 1: Bag of Words (Saco de Palavras)

O **Bag of Words (BoW)** é uma das técnicas mais clássicas e diretas para converter texto em números. A intuição por trás do método é exatamente o que o nome sugere: o modelo pega todas as palavras do texto, "joga dentro de um saco" e descarta totalmente a estrutura gramatical, a ordem das frases e o contexto. O que importa para o algoritmo é apenas a **presença e a frequência** das palavras.

#### ⚙️ Como funciona o fluxo técnico:
1. **Vocabulário Global:** O algoritmo varre todo o conjunto de texto de treino e cria um dicionário com todas as palavras únicas encontradas.
2. **Criação de Colunas:** Cada palavra desse dicionário vira uma coluna em nossa matriz de features ($X$).
3. **Vetorização (Contagem):** Para cada avaliação, o modelo preenche as colunas contando quantas vezes aquela palavra específica apareceu na frase.

#### 📊 Exemplo Prático:
Se tivéssemos apenas duas avaliações na base de dados:
* *Avaliação 1:* `"Ótimo produto, ótimo prazo."`
* *Avaliação 2:* `"Produto ruim."`

O resultado que o modelo de Machine Learning recebe é uma tabela numérica parecida com esta:


| Avaliação | ótimo | produto | prazo | ruim |
| :--- | :---: | :---: | :---: | :---: |
| **Avaliação 1** | 2 | 1 | 1 | 0 |
| **Avaliação 2** | 0 | 1 | 0 | 1 |

#### ⚖️ Prós e Contras da Técnica:
* **Vantagens:** Extremamente simples de entender, rápida de computar e muito eficiente para identificar palavras de forte impacto (como "excelente" para positivo ou "horroroso" para negativo).
* **Limitações (O Ponto Cego):** Não entende contexto ou negações. Para o Bag of Words, as frases *"O produto é bom, não é ruim"* e *"O produto é ruim, não é bom"* geram exatamente os mesmos números para o saco, apesar de terem sentimentos completamente opostos.


### **3.2.1. Vetorização**

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
import scipy.sparse as sp

In [ ]:
# Vetorizar apenas o texto da avaliação
print("--- Executando TÉCNICA 1: Bag of Words + Nota Numérica ---")

bow_vectorizer = CountVectorizer(max_features=50)
X_train_vector = bow_vectorizer.fit_transform(X_train)
X_test_vector = bow_vectorizer.transform(X_test)

In [ ]:
X_train_vector.shape

### **3.2.2. Treinando o modelo**

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

In [ ]:
# Logistic Regression
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_vector, y_train)

In [ ]:
predicoes_bag_lr = lr.predict(X_test_vector)

### **3.2.3. Avaliação**

In [ ]:
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay, classification_report, confusion_matrix

In [ ]:
# 📊 MÉTRICAS E RESULTADOS Logistic Regression
# ==========================================
import matplotlib.pyplot as plt

print(f"\nAcurácia Geral: {accuracy_score(y_test, predicoes_bag_lr):.4f}")
print('---'*20)
print("\nRelatório de Classificação Detalhado:")
print(classification_report(y_test, predicoes_bag_lr))
print('---'*20)
ConfusionMatrixDisplay.from_estimator(lr, X_test_vector, y_test)
plt.ylabel('Classe Real')
plt.xlabel('Classe Prevista')
plt.grid(False)
plt.show()

## **3.3. TF-IDF (Term Frequency-Inverse Document Frequency)**

### 📊 Técnica 2: TF-IDF (Term Frequency-Inverse Document Frequency)

O **TF-IDF** é uma evolução matemática direta do Bag of Words. Enquanto o método anterior apenas conta a frequência absoluta das palavras, o TF-IDF calcula a **relevância e a importância real** de cada palavra dentro do seu conjunto de dados. Ele foi desenhado para resolver o problema de palavras muito comuns que não agregam valor à classificação.

#### ⚙️ Como funciona a fórmula matemática:
O TF-IDF é o resultado da multiplicação de dois componentes principais:
1. **TF (Term Frequency / Frequência do Termo):** Mede quão frequente uma palavra é em uma avaliação específica. Se a palavra aparece muito naquela linha, o TF aumenta.
2. **IDF (Inverse Document Frequency / Frequência Inversa nos Documentos):** Mede quão rara a palavra é ao olhar para a base inteira. Palavras que aparecem em quase todas as avaliações (como "o", "a", "produto", "que") recebem um peso IDF próximo de **zero**. Palavras raras e expressivas (como "excelente", "defeito", "maravilhoso") ganham um peso IDF **alto**.

#### ⚖️ Prós e Contras da Técnica:
* **Vantagens:** Consegue filtrar automaticamente o ruído do texto sem a necessidade obrigatória de uma lista gigante de *stop words*. Dá pesos muito precisos para termos que definem o sentimento (positivos ou negativos), sendo o algoritmo clássico de melhor custo-benefício em PLN.
* **Limitações:** Assim como o Bag of Words, o TF-IDF padrão ainda trata as palavras de forma isolada. Ele falha em capturar a ordem exata da frase, o contexto semântico profundo e nuances complexas como o sarcasmo.


### **3.3.1. Vetorização**

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
print("--- Executando TÉCNICA 2: TF-IDF APENAS com Texto (Sem Nota) ---")

# Inicializar o vetorizador TF-IDF
# max_features=5000 limita o vocabulário para evitar lentidão
# sublinear_tf=True aplica escala logarítmica à frequência das palavras, o que melhora o desempenho
tfidf_vectorizer = TfidfVectorizer(max_features=1000, sublinear_tf=True, ngram_range=(1,3))

# Transformar os textos de treino e teste
X_train_tf = tfidf_vectorizer.fit_transform(X_train)
X_test_tf = tfidf_vectorizer.transform(X_test)

print("--- Execução Finalizada ... ---")

### **3.3.2. Treinando os dados**

In [ ]:
from sklearn.linear_model import LogisticRegression

In [ ]:
lr_tf = LogisticRegression(max_iter=1000)
lr_tf.fit(X_train_tf, y_train)

In [ ]:
previsao_lr_tf = lr_tf.predict(X_test_tf)

### **3.3.3. Avaliando o Modelo**

In [ ]:
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay, classification_report, confusion_matrix

In [ ]:
# 📊 MÉTRICAS E RESULTADOS Logistic Regression
# ==========================================
import matplotlib.pyplot as plt

print(f"\nAcurácia Geral: {accuracy_score(y_test, previsao_lr_tf):.4f}")
print('---'*20)
print("\nRelatório de Classificação Detalhado:")
print(classification_report(y_test, previsao_lr_tf))
print('---'*20)
ConfusionMatrixDisplay.from_estimator(lr_tf, X_test_tf, y_test, normalize='true')
plt.ylabel('Classe Real')
plt.xlabel('Classe Prevista')
plt.grid(False)
plt.show()

### **3.3.4. Validação Cruzada**

In [ ]:
from sklearn.model_selection import cross_val_score, KFold

In [ ]:
## Validação Cruzada
from sklearn.model_selection import cross_val_score, KFold

def validacao_cruzada(n_splits=5, cv=5):
    kfold = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    scores = cross_val_score(lr_tf, X_train_tf, y_train, cv=cv)
    return scores

In [ ]:
# Intervalo de Confiança
from scipy.stats import sem, t
import numpy as np

def confidence_interval(data, confidence=0.95):
    n = len(data)
    mean = np.mean(data)
    std_err = np.std(data)
    h = std_err * t.ppf((1 + confidence) / 2, n - 1)
    ic_inferior = mean - h
    ic_superior = mean + h

    print(f"Média: {mean:.4f}\
            Desvio Padrão: {std_err:.4f}\
            Intervalo Confiança: [{ic_inferior:.4f} ~ {ic_superior:.4f}]")

In [ ]:
scores_5 = validacao_cruzada(n_splits=5, cv=5)
print(f"Resultados:\n{scores_5}")
confidence_interval(scores_5, confidence=0.95)

In [ ]:
scores_10 = validacao_cruzada(n_splits=10, cv=10)
print(f"Resultados:\n{scores_10}")
confidence_interval(scores_10, confidence=0.95)

In [ ]:
scores_30 = validacao_cruzada(n_splits=30, cv=30)
print(f"Resultados:\n{scores_30}")
confidence_interval(scores_30, confidence=0.95)

#### **Análise Técnica da Validação Cruzada**

A validação cruzada (K-Fold) nos permite avaliar a **consistência** do modelo em diferentes partes do dataset.

**O que os resultados nos dizem?**
*   **Estabilidade:** Com uma média de acurácia em torno de **93%** e um desvio padrão baixo, o modelo mostra que não sofre de *overfitting* severo e que o desempenho é estável independente da fatia de dados utilizada.
*   **Intervalo de Confiança:** O intervalo calculado (ex: [91.3% ~ 94.8%]) nos dá a segurança estatística de que, ao receber novos dados similares em produção, a acurácia do modelo provavelmente se manterá dentro desta faixa.
*   **Confiabilidade de Negócio:** Isso demonstra que é uma solução robusta. Se o desvio padrão fosse muito alto, saberíamos que o modelo é instável e dependente de exemplos específicos.

### **3.3.5. Salvando o modelo**

In [ ]:
import joblib
joblib.dump(lr_tf, '/content/drive/MyDrive/models-pkl/model_logistic_regressor_nlp_sentimentos.pkl')
joblib.dump(tfidf_vectorizer, '/content/drive/MyDrive/models-pkl/tfidf_vectorizer_nlp_sentimentos.pkl')

## **3.4. Word Embeddings (Word2Vec)**

### 🌐 Técnica 3: Word Embeddings (Word2Vec / GloVe)

Os **Word Embeddings** representam uma mudança de paradigma em relação às abordagens estatísticas anteriores (BoW e TF-IDF). Em vez de tratar as palavras como chaves isoladas em uma tabela, esta técnica mapeia cada palavra para um espaço vetorial contínuo (geralmente de 100 a 300 dimensões).

A grande sacada matemática aqui é que **palavras que aparecem em contextos semelhantes possuem vetores semelhantes**.

#### ⚙️ Como funciona a inteligência da frase:
1. **Significado Geométrico:** No Word2Vec, a distância entre vetores reflete a proximidade de significado. O modelo entende equações conceituais como: $Vetor(Rei) - Vetor(Homem) + Vetor(Mulher) \approx Vetor(Rainha)$.
2. **Vetorização da Frase:** Para classificar uma avaliação inteira, o código extrai o vetor de cada palavra isolada do comentário e calcula o **Vetor Médio** do texto, gerando um mapa consolidado do assunto para o classificador.

#### ⚖️ Prós e Contras da Técnica:
* **Vantagens:** O modelo passa a compreender sinônimos. Se ele aprender no treino que a palavra "péssimo" indica uma insatisfação, ele automaticamente associará a palavra "horrendo" ao mesmo sentimento no teste, mesmo que ela nunca tenha aparecido nos dados de treino.
* **Limitações:** A média simples dos vetores achata o texto e pode perder o foco principal. Além disso, os embeddings tradicionais são **estáticos**: a palavra "banco" terá exatamente o mesmo vetor numérico se a frase for "sentei no banco da praça" ou "paguei o boleto no banco".


### **3.4.1. Bibliotecas**

In [ ]:
!pip install spacy
!python -m spacy download pt_core_news_md

### **3.4.2. Vetorização**

In [ ]:
import numpy as np
import spacy

In [ ]:
print("--- Executando TÉCNICA 3: Word Embeddings (Conceito Word2Vec) ---")

# 1. Carregar o modelo de linguagem em português com suporte a vetores (embeddings)
nlp = spacy.load("pt_core_news_md")

In [ ]:
# 2. Função para transformar um texto completo em um vetor numérico médio
# O Spacy faz exatamente o papel do Word2Vec: calcula a média dos vetores de cada palavra da frase
def extrair_vetor_medio(textos):
    vetores = []
    for doc in nlp.pipe(textos, disable=["ner", "parser"]):
        # doc.vector extrai a média geométrica dos embeddings de todas as palavras da avaliação
        vetores.append(doc.vector)
    return np.array(vetores)

In [ ]:
# 3. Converter os textos de treino e teste em matrizes de vetores (embeddings)
print("Convertendo os textos de treino em vetores...")
X_train_w2v = extrair_vetor_medio(X_train)

In [ ]:
print("Convertendo os textos de teste em vetores...")
X_test_w2v = extrair_vetor_medio(X_test)

### **3.4.3. Treinando o modelo**

In [ ]:
lr_we = LogisticRegression(max_iter=1000)

lr_we.fit(X_train_w2v, y_train)

In [ ]:
previsao_we = lr_we.predict(X_test_w2v)

### **3.4.4. Avaliação**

In [ ]:
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay, classification_report, confusion_matrix

print(f"\nAcurácia Geral: {accuracy_score(y_test, previsao_we):.4f}\n")
print("\nRelatório de Classificação Detalhado:")
print(classification_report(y_test, previsao_we))
ConfusionMatrixDisplay.from_estimator(lr_we, X_test_w2v, y_test)
plt.ylabel('Classe Real')
plt.xlabel('Classe Prevista')
plt.grid(False)
plt.show()

## **3.5. Transformers (BERTimbau)**

Em vez de tirar uma média matemática burra das palavras, os Transformers usam o mecanismo de Atenção. O BERT lê a frase inteira e consegue entender que a palavra "defeito" é o núcleo principal da avaliação, dando um peso gigantesco para ela no vetor final ([CLS]).

### 🤖 Técnica 4: Modelos Baseados em Transformers (BERTimbau)

Os **Transformers** representam o estado da arte absoluto em Processamento de Linguagem Natural (PLN) e são o motor tecnológico por trás dos sistemas de busca modernos, LLMs (como GPT e Claude) e mecanismos de **RAG**. Nesta etapa, utilizamos o **BERTimbau**, um modelo BERT pré-treinado massivamente com textos em português do Brasil [1].

#### ⚙️ Como funciona a revolução da Atenção Contextual:
1. **Embeddings Dinâmicos:** Ao contrário do Word2Vec, o BERT gera números diferentes para a mesma palavra dependendo do contexto. Ele analisa a frase inteira simultaneamente por meio do mecanismo de *Self-Attention*, capturando dependências de longo alcance no texto.
2. **O Token [CLS]:** No início de cada texto enviado ao BERT, o modelo insere um token especial chamado `[CLS]` (Classification). O vetor numérico gerado para este token específico funciona como uma "assinatura digital compressa" que resume o sentido e o sentimento da frase inteira de forma semântica profunda.

#### ⚖️ Prós e Contras da Técnica:
* **Vantagens:** Máxima precisão possível em tarefas de PLN. Compreende perfeitamente ironias, inversões sintáticas, sinônimos distantes e nuances gramaticais complexas. É a tecnologia ideal para aplicações robustas de mercado.
* **Limitações:** Alto custo computacional. Exige o uso de GPUs para processamento em tempo hábil e gera vetores densos grandes, exigindo mais memória tanto no treinamento quanto na inferência em produção.


### **3.5.1. Bibliotecas**

In [ ]:
import torch
import numpy as np
from tqdm import tqdm
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from transformers import AutoTokenizer, AutoModel

### **3.5.2. Vetorização**

In [ ]:
print("--- Executando TÉCNICA 4: BERTimbau (Transformers) ---")

# 1. Carregar o Tokenizer e o Modelo BERTimbau (Oficial para Português)
# Usamos a versão 'base' que roda bem em ambientes de estudo
tokenizer = AutoTokenizer.from_pretrained('neuralmind/bert-base-portuguese-cased')
bert_model = AutoModel.from_pretrained('neuralmind/bert-base-portuguese-cased')

# Mover o modelo para a GPU se ela estiver disponível
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
bert_model = bert_model.to(device)

In [ ]:
# 2. Função para extrair os Embeddings de Contexto (idêntico ao conceito do RAG)
def extrair_embeddings_contextuais(textos):
    vetores = []
    bert_model.eval() # Coloca o modelo em modo de avaliação (desativa dropout)

    with torch.no_grad(): # Desativa o cálculo de gradientes para ir mais rápido
        for texto in tqdm(textos, desc="Gerando vetores inteligentes no BERT"):
            # O Tokenizer quebra o texto e adiciona os marcadores especiais do BERT (como o [CLS])
            inputs = tokenizer(str(texto), padding=True, truncation=True, max_length=128, return_tensors="pt").to(device)
            outputs = bert_model(**inputs)

            # Pegamos o vetor do token [CLS] (índice 0), que representa o resumo contextual da frase inteira
            vetor_frase = outputs.last_hidden_state[:, 0, :].cpu().numpy().flatten()
            vetores.append(vetor_frase)

    return np.array(vetores)

In [ ]:
# 3. Transformar os textos de treino e teste em vetores do BERT
# (Isso pode demorar um pouco dependendo do tamanho da sua base)
X_train_bert = extrair_embeddings_contextuais(X_train)
X_test_bert = extrair_embeddings_contextuais(X_test)

### **3.5.3. Treinando o Modelo**

In [ ]:
# 4. Treinar o mesmo classificador Random Forest nos dados do BERT
modelo_bert = LogisticRegression(max_iter=1000)
modelo_bert.fit(X_train_bert, y_train)

# 5. Previsões e Métricas finais
preds_bert = modelo_bert.predict(X_test_bert)

### **3.5.4. Avaliação**

In [ ]:
print(f"\nAcurácia Geral (BERTimbau): {accuracy_score(y_test, preds_bert):.4f}\n")
print("\nRelatório de Classificação Detalhado:")
print(classification_report(y_test, preds_bert))
print("Matriz de Confusão:")
ConfusionMatrixDisplay.from_estimator(modelo_bert, X_test_bert, y_test)
plt.ylabel('Classe Real')
plt.xlabel('Classe Prevista')
plt.grid(False)
plt.show()

### **3.5.5. Salvando o Modelo BERT e o Tokenizer**

In [ ]:
# Salvando os embeddings do BERT
np.save('/content/drive/MyDrive/models-pkl/X_train_bert_embeddings.npy', X_train_bert)
np.save('/content/drive/MyDrive/models-pkl/X_test_bert_embeddings.npy', X_test_bert)
print('Embeddings do BERT para treino e teste salvos com sucesso!')

In [ ]:
import joblib
joblib.dump(modelo_bert, '/content/drive/MyDrive/models-pkl/bert_model_nlp_sentimentos.pkl')
joblib.dump(tokenizer, '/content/drive/MyDrive/models-pkl/bert_tokenizer_nlp_sentimentos.pkl')
print('Modelo BERT e Tokenizer salvos com sucesso!')

In [ ]:
# Salva o tokenizer e o BERTimbau na pasta local do projeto
tokenizer.save_pretrained('/content/drive/MyDrive/models-pkl')
bert_model.save_pretrained('/content/drive/MyDrive/models-pkl')

### **3.5.6. Lendo o Modelo BERT e o Tokenizer**

In [ ]:
# 1. Carrega o classificador de ML (arquivo .pkl)
classificador = joblib.load('/content/drive/MyDrive/models-pkl/bert_model_nlp_sentimentos.pkl')

# 2. Carrega o BERTimbau apontando para a PASTA que foi criada no Drive
pasta_bert_drive = '/content/drive/MyDrive/models-pkl/bert_model_nlp_sentimentos'
tokenizer = AutoTokenizer.from_pretrained('/content/drive/MyDrive/models-pkl')
bert_model = AutoModel.from_pretrained('/content/drive/MyDrive/models-pkl')


In [ ]:
# 2. FUNÇÃO PARA VETORIZAR OS NOVOS TEXTOS
def extrair_embeddings_novos(textos):
    vetores = []
    bert_model.eval()
    with torch.no_grad():
        for texto in textos:
            inputs = tokenizer(str(texto), padding=True, truncation=True, max_length=128, return_tensors="pt").to(device)
            outputs = bert_model(**inputs)
            # Extrai o token [CLS] (índice 0)
            vetor_frase = outputs.last_hidden_state[:, 0, :].cpu().numpy().flatten()
            vetores.append(vetor_frase)
    return np.array(vetores)

In [ ]:
def previsao_sentimento_novo(textos_novos):
    # Aplica a vetorização
    text = extrair_embeddings_novos(textos_novos)
    # Faz a predição
    result = classificador.predict(text)
    return result

In [ ]:
text = ['O produto chegou antes do prazo e funciona perfeitamente!']
text2 = ['O produto é ruim de péssima qualidade']
result_previsao = previsao_sentimento_novo(text2)
if result_previsao == 1:
    print('O Sentimento do cliente: POSITIVO!')
else:
    print('O Sentimento do cliente: NEGATIVO !')

# **<font color='black'> 4. Conclusão e Comparativo de Resultados**

Após testar diferentes abordagens de Processamento de Linguagem Natural, chegamos aos seguintes resultados de acurácia aproximada:

| Técnica | Modelo | Acurácia | Observação Principal |
| :--- | :--- | :---: | :--- |
| **Bag of Words** | Logistic Regression | ~88% | Simples e rápido, mas ignora o contexto das palavras. |<font color='blue'>
|<font color='blue'> **TF-IDF** </font>|<font color='blue'> Logistic Regression </font>|<font color='blue'> **~93%** </font>|<font color='blue'> Excelente equilíbrio entre custo computacional e performance.</font>|
| **Word Embeddings** | Logistic Regression | ~88% | Captura similaridade semântica, mas a média dos vetores pode perder nuances. |
| **BERTimbau** | Logistic Regression | ???? | Estado da arte, entende o contexto profundo, porém exige muito mais poder de processamento. |

### **Análise Final**
1.  **O Vencedor Prático:** O **TF-IDF** apresentou o melhor desempenho neste conjunto de dados específico. Isso ocorre porque, em avaliações de e-commerce, palavras-chave específicas (como "defeito", "amei", "atraso") são indicadores muito fortes de sentimento, e o TF-IDF consegue dar o peso correto a elas.
2.  **Modelos Complexos:** Embora o **BERTimbau** seja tecnicamente superior, sua vantagem competitiva brilha em textos mais longos e complexos. Para frases curtas de avaliações, modelos lineares com TF-IDF costumam ser extremamente robustos.
3.  **Próximos Passos:** Para evoluir este projeto, poderíamos realizar um *Fine-tuning* do BERT ou testar técnicas de balanceamento de classes caso a base de dados fosse mais desproporcional.

# **<font color='blue'> 5. Prevendo novos valores**

## **5.1. Bibliotecas**

In [ ]:
!pip install unidecode
nltk.download('rslp')
nltk.download('stopwords')

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
import nltk
import joblib
from nltk import tokenize
import unidecode
from nltk.corpus import stopwords
from nltk.stem import RSLPStemmer


## **5.2. Carregando modelos**

In [ ]:
modelo_ml = joblib.load('/content/drive/MyDrive/models-pkl/model_logistic_regressor_nlp_sentimentos.pkl')
tfidf_vectorizer = joblib.load('/content/drive/MyDrive/models-pkl/tfidf_vectorizer_nlp_sentimentos.pkl')

## **5.3. Funções**

In [ ]:
palavras_irrelevantes = stopwords.words('portuguese')
token = tokenize.WordPunctTokenizer()
stemmer = RSLPStemmer()

def limpeza_dados(texto):

    # Passo 1: Deixar tudo em letras minusculas
    texto = texto.lower()
    # Passo 2: Remover acentuação
    texto = unidecode.unidecode(texto)
    # Passo 3: Remover caracteres não alpha
    texto = token.tokenize(texto)
    texto = [palavra for palavra in texto if palavra.isalpha()]
    # Passo 4: Remover palavras irrelevantes
    texto = [palavra for palavra in texto if palavra not in palavras_irrelevantes]
    # Passo 5: Reduzir cada uma delas ao seu radical (raiz)
    texto = [stemmer.stem(palavra) for palavra in texto]
    texto = ' '.join(texto)

    return texto

In [ ]:
texto_teste = 'Produto de péssima qualidade. Apresentou defeito e não funciona corretamente'
texto_limpo = limpeza_dados(texto_teste)
print(f"Texto Original: {texto_teste}")
print(f"Texto Limpo: {texto_limpo}")

In [ ]:
def previsao_sentimento(text):
    # Aplica a vetorização
    text = tfidf_vectorizer.transform([text])
    # Faz a predição
    result = modelo_ml.predict(text)
    return result

In [ ]:
def formatacao_texto(texto):
    print("Fazendo previsão do sentimento do cliente...")
    print('---'*20)
    print(f"Texto Original: {texto}")

    # Faz a limpeza do texto
    texto_limpo = limpeza_dados(texto)
    print(f"Texto tratado: {texto_limpo}")

    # Faz a previsão do sentimento
    previsao = previsao_sentimento(texto_limpo)

    print('==='*20)
    print("Previsão:")
    if previsao == 1:
        print('O Sentimento do cliente: POSITIVO!')
    else:
        print('O Sentimento do cliente: NEGATIVO !')

## **5.4. Fazendo Previsões de Sentimentos**

In [ ]:
texto = 'O produto apresentou defeito. Não recomendo!'
formatacao_texto(texto)

In [ ]:
texto_2 = 'O produto é mais ou menos. Comprarei novamente!'
formatacao_texto(texto_2)

In [ ]:
texto_3 = 'Nesse produto eu dou nota ótimo'
formatacao_texto(texto_3)